# Week 02 - Mini Semantic Search System

This notebook demonstrates building a mini semantic search system using pandas, sentence-transformers, and ChromaDB.

## 1. Install dependencies

In [ ]:
pip install -r requirements.txt

: 

## 2. Import libraries

In [ ]:
import pandas as pd
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

## 3. Load and inspect FAQ data

In [ ]:
DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'faqs.csv')
df

## 4. Clean and prepare chunks

In [ ]:
df['question'] = df['question'].astype(str).str.strip()
df['answer'] = df['answer'].astype(str).str.strip()
df['chunk'] = df['question'] + ' ' + df['answer']
df

## 5. Generate embeddings and store in ChromaDB

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path='./chroma_db')
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

try:
    client.delete_collection(name='faqs_semantic_search')
except Exception:
    pass

collection = client.create_collection(name='faqs_semantic_search', embedding_function=embedding_fn)

documents = df['chunk'].tolist()
metadatas = [{'question': row['question'], 'answer': row['answer']} for _, row in df.iterrows()]
ids = [f'faq_{i}' for i in range(len(df))]

collection.add(documents=documents, metadatas=metadatas, ids=ids)
print('Index built.')

## 6. Perform semantic search

In [ ]:
query = 'What is machine learning?'
results = collection.query(query_texts=[query], n_results=3)
results

In [ ]:
docs = results['documents'][0]
metas = results['metadatas'][0]
dists = results['distances'][0]
for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
    print(f'Top {i}:')
    print('Question:', meta['question'])
    print('Answer:', meta['answer'])
    print('Distance:', dist)
    print()